# HOPE
- evaluation metric  based on information preservation 𝜁𝑖𝑛𝑓 , semantic independence 𝜁sem and concept unity 𝜁con, all divided by 3.

I will be running this on a sample rather than full corpus to save resources. I will determine the amount of samples n such that I have a 99 % confidence.
n = (Z² × σ² × N) / (E² × (N-1) + Z² × σ²). or actually, i might do a paired test because it takes fewer samples and is also rigorous. 

In [ ]:
N = 879
Z = 2.576  ##99% conf



In [ ]:
# Imports
import ollama
import json
import random
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np
import random
import time
import re

random.seed(42)

embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
# env variables
PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"
#Chunk folders
recursive_1000 = DATA_DIR / "chunks" / "recursive_1000"
rsc = DATA_DIR / "chunks" / "rsc"
semantic_p95 = DATA_DIR / "chunks" / "semantic_p95"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1828.17it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## ζ_con (Concept Unity)
- For each chunk, send it to an LLM → get a set of statements about what the chunk contains
- Embed each statement
- Compute pairwise cosine similarity between all statement embeddings
- Average = that chunk's concept unity score
- Average across all chunks = ζ_con

In [12]:
import numpy as np
from itertools import combinations

def get_statements(chunk_text, model="gemma3:1b", num_statements=5):
    #Asks LLM to extract factual statements from a chunk
    prompt = (
        f"Extract exactly {num_statements} distinct factual statements from the following scientific text. "
        "Return ONLY a JSON list of strings. No explanation, no markdown.\n\n"#problematic
        f"Text: {chunk_text}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.7} # in the paper, they said non zero temperature
    )
    raw = response["message"]["content"]
    
    # Clean up common LLM output issues
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1]  # remove first line
        raw = raw.rsplit("```", 1)[0]  # remove closing fence
    
    statements = json.loads(raw)
    return statements

In [13]:
def concept_unity_chunk(statements, embed_model):
    """Compute concept unity score for a single chunk's statements."""
    if len(statements) < 2:
        return 0.0
    
    # Embed all statements
    embeddings = embed_model.embed_documents(statements)
    embeddings = np.array(embeddings)
    
    # Pairwise cosine similarity (following paper: includes self-pairs, divides by |S|^2)
    n = len(embeddings)
    total_sim = 0.0
    for i in range(n):
        for j in range(n):
            sim = np.dot(embeddings[i], embeddings[j]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j])
            )
            total_sim += sim
    
    score = total_sim / (n * n)
    return max(score, 0.0)  # clamp negatives to 0

In [14]:
def ζ_con(chunks, embed_model, sample_n=10, llm_model="granite3.2:8b", num_statements=5):
    """
    Compute ζ_con for a list of chunk dicts.
    Subsamples sample_n chunks if there are more.
    Returns dict with average score and details.
    """
    if len(chunks) > sample_n:
        sampled = random.sample(chunks, sample_n)
    else:
        sampled = chunks

    scores = []
    failed = 0
    for chunk in sampled:
        try:
            statements = get_statements(chunk["text"], model=llm_model, num_statements=num_statements)
            score = concept_unity_chunk(statements, embed_model)
            scores.append({"chunk_id": chunk["chunk_id"], "score": score})
        except (json.JSONDecodeError, ValueError, KeyError) as e:
            failed += 1
            print(f"  Chunk {chunk['chunk_id']} failed: {e}")
            continue

    avg = float(np.mean([s["score"] for s in scores])) if scores else 0.0
    return {"avg": avg, "n_evaluated": len(scores), "n_failed": failed, "chunks": scores}

In [11]:
# Quick test — loads one doc, try one chunk
test_path = recursive_1000 / "algae-2016-31-12-9.json"
with open(test_path, encoding = "utf-8") as f:
    test_doc = json.load(f)

# See what granite actually outputs
response = ollama.chat(
    model="granite3.2:8b",
    messages=[{"role": "user", "content": 
        "Extract exactly 5 distinct factual statements from the following scientific text. "
        "Return ONLY a JSON list of strings. No explanation, no markdown, no code fences.\n\n"
        f"Text: {test_doc['chunks'][0]['text']}"
    }],
    options={"temperature": 0.7},
)
raw = response["message"]["content"]
print("RAW OUTPUT:")
print(repr(raw))

RAW OUTPUT:
'["Cyr Abel Maranguy Ogandaga, Han Gil Choi, Jang Kyun Kim, and Ki Wan Nam are authors of this study.",\n"The study is about growth responses of Chondrus ocellatus to two endophytes.",\n"The two endophytes are Mikrosyphar zosterae and Ulvella ramosa.",\n"Mikrosyphar zosterae belongs to the class Ectocarpales, order Ochrophyta.",\n"Ulvella ramosa belongs to the class Ulvales, order Chlorophyta."]'


In [15]:
# Sample 30 docs present in all 3 strategies
all_files = (
    {f.name for f in recursive_1000.glob("*.json")}
    & {f.name for f in semantic_p95.glob("*.json")}
    & {f.name for f in rsc.glob("*.json")}
)
sampled_docs = sorted(random.sample(sorted(all_files), 30))
print(f"Sampled {len(sampled_docs)} docs from {len(all_files)} common files")

# Results dict — saves after every doc/strategy
results_path = PROJECT_ROOT / "outputs" / "HOPE" /"hope_con_results.json"

if results_path.exists():
    with open(results_path, encoding="utf-8") as f:
        results = json.load(f)
    print(f"Resuming — {len(results)} entries already done")
else:
    results = {}

strategies = {
    "recursive_1000": recursive_1000,
    "semantic_p95": semantic_p95,
    "rsc": rsc,
}

total = len(sampled_docs) * len(strategies)
done = 0

for doc_name in sampled_docs:
    for strat_name, strat_path in strategies.items():
        key = f"{strat_name}/{doc_name}"
        if key in results:
            done += 1
            continue

        with open(strat_path / doc_name, encoding="utf-8") as f:
            doc = json.load(f)

        print(f"[{done+1}/{total}] {key} ({doc['num_chunks']} chunks)")
        start = time.time()
        result = ζ_con(doc["chunks"], embed_model)
        elapsed = time.time() - start

        result["elapsed_s"] = round(elapsed, 1)
        results[key] = result
        print(f"  ζ_con={result['avg']:.4f} | {result['n_evaluated']} passed, {result['n_failed']} failed | {elapsed:.0f}s")

        # Save after every doc/strategy
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        done += 1

print("\nDone! Results saved in:", results_path)

Sampled 30 docs from 868 common files
[1/90] recursive_1000/algae-1986-1-1-241.json (20 chunks)
  ζ_con=0.6831 | 10 passed, 0 failed | 933s
[2/90] semantic_p95/algae-1986-1-1-241.json (9 chunks)
  Chunk 0 failed: Expecting value: line 1 column 1 (char 0)
  ζ_con=0.6249 | 8 passed, 1 failed | 1601s
[3/90] rsc/algae-1986-1-1-241.json (18 chunks)
  Chunk 2 failed: Expecting ',' delimiter: line 4 column 47 (char 225)
  Chunk 4 failed: Expecting ',' delimiter: line 5 column 634 (char 1164)
  ζ_con=0.7149 | 8 passed, 2 failed | 1370s
[4/90] recursive_1000/algae-1992-7-1-147.json (31 chunks)
  ζ_con=0.7586 | 10 passed, 0 failed | 644s
[5/90] semantic_p95/algae-1992-7-1-147.json (9 chunks)
  Chunk 0 failed: Expecting ',' delimiter: line 1 column 61 (char 60)
  ζ_con=0.7542 | 8 passed, 1 failed | 1506s
[6/90] rsc/algae-1992-7-1-147.json (22 chunks)
  ζ_con=0.7548 | 10 passed, 0 failed | 822s
[7/90] recursive_1000/algae-1997-12-4-269.json (26 chunks)
  ζ_con=0.7352 | 10 passed, 0 failed | 1128s


In [ ]:
import pandas as pd # saving results of each metric and cuhnking sterategy as a 3x3 matrix

# Load ζ_con results
with open(PROJECT_ROOT / "outputs" / "HOPE" / "hope_con_results.json", encoding="utf-8") as f:
    ζ_con_results = json.load(f)

# Aggregate per strategy
strategies = ["recursive_1000", "semantic_p95", "rsc"]
ζ_con = {
    strategy: np.mean([
        ζ_con_results[key]["avg"]
        for key in ζ_con_results
        if key.startswith(strategy + "/")
    ])
    for strategy in strategies
}

# Placeholder until computed
ζ_sem = {s: np.nan for s in strategies}
ζ_inf = {s: np.nan for s in strategies}

# Build the dataframe
hope_df = pd.DataFrame({
    "ζ_con": ζ_con,
    "ζ_sem": ζ_sem,
    "ζ_inf": ζ_inf,
})
hope_df["HOPE"] = hope_df.mean(axis=1)
hope_df

,ζ_con,ζ_sem,ζ_inf,HOPE
recursive_1000,0.726836,NaN,NaN,0.726836
semantic_p95,0.716204,NaN,NaN,0.716204
rsc,0.728889,NaN,NaN,0.728889


## ζ_inf (Information Preservation)
- Randomly sample 3-sentence segments from the ORIGINAL document (not chunks)
- For each segment, have an LLM generate 1 true statement + 3 plausible false ones
- Use the true statement to retrieve relevant chunks from the vector store
- Have an LLM pick which of the 4 statements is true, using only the retrieved chunks
- If it picks correctly, the information was preserved. Binary 1/0 scoring
- Average across all samples = ζ_inf

In [28]:
def clean_sentences(doc_text):
    """Split document text into clean prose sentences, filtering junk."""
    raw_sentences = re.split(r'(?<=[.!?])\s+', doc_text)
    sentences = []
    for s in raw_sentences:
        s = s.strip()
        if len(s) < 40:
            continue
        if not re.match(r'[A-Z]', s):
            continue
        if s.count(' ') < 5:
            continue
        if '\n' in s:
            continue
        if re.search(r'\d+\.\d+\s*[+±]\s*\d+', s):  # measurements
            continue
        if re.search(r'(.)\1{4,}', s):  # repeated characters (OCR garbage)
            continue
        sentences.append(s)
    return sentences

In [29]:
def sample_segments(sentences, num_segments=5, segment_length=3):
    """Pick random consecutive 3-sentence segments from a list of sentences."""
    if len(sentences) < segment_length:
        return []
    
    max_start = len(sentences) - segment_length
    starts = random.sample(range(max_start + 1), min(num_segments, max_start + 1))
    
    segments = [" ".join(sentences[start:start + segment_length]) for start in starts]
    return segments

In [31]:
# test on oyendo islands zostera marina paper turned out poor
test_path = PROJECT_ROOT / "data" / "test" / "algae-2020-35-5-31.json"
with open(test_path, encoding="utf-8") as f:
    doc = json.load(f)

sentences = clean_sentences(doc["text"])
print(f"Clean sentences: {len(sentences)}")
print(f"First 3: {sentences[:3]}")
print()

segments = sample_segments(sentences)
print(f"Segments sampled: {len(segments)}")
for i, seg in enumerate(segments):
    print(f"\nSegment {i}: {seg[:200]}...")

Clean sentences: 80
First 3: ['West3, Susan Loiseaux-de Goër4, Phaik-Eem Lim5, Ahemad Sade6, Hannah Russell7 and Frithjof C.', 'Molecular data (rbcL, large subunit ribosomal DNA, and cytochrome c oxidase subunit I) were obtained from these strains and evidence presented to recognize the new species: Hypoglossum sabahense from Sabah, Malaysia.', 'Because various aspects of morphology in culture speci- mens differ significantly from types based on field specimens we have to rely mainly on the molecular criteria in as- cribing a new taxonomic name here.']

Segments sampled: 5

Segment 0: Mutation rate is easily affected by ef- fective population size, the degree of selection pressure, reproductive system, etc. West shows an extremely long branch in the LSU tree, mean- ing it has a muc...

Segment 1: Strains 4731 and 4732 originated from 2 different samples that were collected at the same site at different depths, while the strain 4731 was collected one day ear- lier. For comparative obser

In [ ]:
def generate_quadruple(segment, model="granite3.2:8b"):
    """Generate 1 true + 3 false statements from a document segment."""
    prompt = (
        "Read the following segment from a scientific paper. "
        "Write exactly 1 true factual statement based on the segment, "
        "and 3 plausible but false statements that sound like they could be from the same paper. "
        "The false statements should be subtle — wrong numbers, swapped species names, or inverted relationships. "
        "Return ONLY a JSON object with keys 'true' and 'false', where 'true' is a string "
        "and 'false' is a list of 3 strings. No explanation, no markdown.\n\n"
        f"Segment: {segment}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.7},
    )
    raw = response["message"]["content"].strip()
    
    if raw.startswith("```"): #sometimes it returns backticks
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
        if "```" in raw:
            raw = raw.rsplit("```", 1)[0]
    raw = raw.strip()
    
    result = json.loads(raw)
    if "true" not in result or "false" not in result:
        raise ValueError(f"Missing keys in response: {result.keys()}")
    if len(result["false"]) != 3:
        raise ValueError(f"Expected 3 false statements, got {len(result['false'])}")
    
    return result

In [35]:
segments = sample_segments(clean_sentences(doc["text"]))
quad = generate_quadruple(segments[0])
print("True:", quad["true"])
print("False:", quad["false"])

True: Rhizoidal filament initials originate from marginal blade cells.
False: ['Rhizoid initial cells contain a maximum of 4 nuclei.', 'Rhizoid filament cells measure 20-30 µm in diameter, and 200-600 µm in length with nuclei evenly distributed.', 'Marginal blade cells are the origin of Rhizoid filament initials, which are 10-15 µm in diameter and 200-500 µm long.']


It seems to work well, the differences are subtle.

In [37]:
def judge_quadruple(quadruple, retrieved_chunks, model="granite3.2:8b"):
    """Given a quadruple and retrieved chunks, ask LLM which statement is true."""
    # Shuffle so the true statement isn't always first
    statements = [quadruple["true"]] + quadruple["false"]
    random.shuffle(statements)
    true_index = statements.index(quadruple["true"])
    
    context = "\n\n".join(retrieved_chunks)
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(statements))
    
    prompt = (
        "Based ONLY on the following context, which of the 4 statements is true? "
        "Reply with ONLY the number (1, 2, 3, or 4). Nothing else.\n\n"
        f"Context:\n{context}\n\n"
        f"Statements:\n{numbered}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.0},  # deterministic for judging
    )
    raw = response["message"]["content"].strip()
    
    # Extract the number from response
    match = re.search(r'[1-4]', raw)
    if not match:
        return False
    
    predicted = int(match.group()) - 1  # 0-indexed
    return predicted == true_index

In [42]:
def ζ_inf_document(doc_text, doc_filename, collection, embed_model, 
                    num_segments=5, model="granite3.2:8b"):
    """Compute information preservation for one document against one collection."""
    sentences = clean_sentences(doc_text)
    segments = sample_segments(sentences, num_segments=num_segments)
    
    if not segments:
        return {"score": np.nan, "n_segments": 0, "details": []}
    
    correct = 0
    details = []
    
    for seg in segments:
        try:
            # Generate quadruple
            quad = generate_quadruple(seg, model=model)
            
            # Retrieve chunks using the true statement as query
            query_embedding = embed_model.embed_query(quad["true"])
            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=3,
                where={"filename": doc_filename}
            )
            
            retrieved = results["documents"][0] if results["documents"][0] else []
            if not retrieved:
                details.append({"segment": seg[:100], "result": "no_chunks_retrieved"})
                continue
            
            # Judge
            is_correct = judge_quadruple(quad, retrieved, model=model)
            correct += int(is_correct)
            details.append({"segment": seg[:100], "result": is_correct})
            
        except (json.JSONDecodeError, ValueError, KeyError) as e:
            details.append({"segment": seg[:100], "result": f"error: {e}"})
            continue
    
    scored = [d for d in details if isinstance(d["result"], bool)]
    score = correct / len(scored) if scored else 0.0
    
    return {"score": score, "n_segments": len(scored), "details": details}

## ζ_sem (Semantic Independence)
- For each chunk p*, send it to an LLM → generate questions that p* should answer
- Answer each question using ONLY p* (completion mode)
- Answer each question using p* PLUS top-3 most similar other chunks (RAG mode)
- Embed both answer sets, compute pairwise cosine similarity between them
- High similarity = the chunk stands on its own. Low = it depends on neighbors
- Average across all chunks = ζ_sem

In [5]:
ζ_inf = ...
ζ_sem = ...
ζ_con = ...
def HOPE(ζ_inf,ζ_sem,ζ_con):
    return (1/3) * (ζ_inf+ζ_sem +ζ_con)

## Recursive evaluation

In [ ]:
CHUNKS_DIR = PROJECT_ROOT / "data" / "chunks" / "recursive_1000"